<a href="https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
1. **Staleness** (`days_since_last_update`) — the signal behind FlyRank's refresh flags. The idea: a page that hasn't been touched in a long time is more likely to be slipping.
2. **CTR-vs-position** (`ctr` by `position_tier`) — the signal behind FlyRank's CTR-fix logic (`needs_ctr_fix`). The idea: a page ranking well should earn clicks in proportion to its position, and one that doesn't is a CTR problem, not a ranking problem.

For the staleness check I compare against `is_declining_label` (`trend_direction == "down"`), computed exactly the way the starter pipeline defines it. This is a **check only** — the label is never an input to the rule below, only something I measure the signal against.


In [1]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/MissNaliaka/SEO-Content-Opportunity-Scoring/refs/heads/main/data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("SIGNAL 1 - Staleness vs decline rate")
staleness_check = (
    df.groupby("freshness_tier")
    .agg(n=("content_id", "size"), decline_rate = ("is_declining_label", "mean"))
    .round(3)
    .reindex(["0-30", "31-90", "91-180", "181+"])
)
print(staleness_check)
print("Verdict: MIXED — real 10pp gap between the two well-populated tiers, but the n=174 tail reverses it.\n")



SIGNAL 1 - Staleness vs decline rate
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471
Verdict: MIXED — real 10pp gap between the two well-populated tiers, but the n=174 tail reverses it.



The two well-populated tiers tell a real story: 0-30 days sits at 51.1% declining, 91-180 days sits at 61.1% — a 10-point gap on tens of thousands of rows combined. But the tail flips it: 181+ days is *lower* than the fresh tier (47.1%), on only n=174 rows. That's too small to trust, and it directly contradicts the "more stale = more risk" story the signal is supposed to tell.

**Verdict: MIXED.** Staleness is real and directional in the well-populated middle of the distribution, but it does not hold at the far tail where a `>=180 days` threshold (closer to FlyRank's own refresh-flag cutoff) would keep firing hardest. I'm using `>=91 days` instead — that's where the honest signal actually lives, not the tail I can't trust.

In [5]:
print("SIGNAL 2 — CTR vs position (behind FlyRank's CTR-fix logic / needs_ctr_fix)")
ctr_check = (
    df[df["position_tier"] != "no_data"]
    .groupby("position_tier")
    .agg(n=("content_id", "size"), mean_ctr=("ctr", "mean"))
    .round(3)
    .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
)
print(ctr_check)
print("Verdict: CONFIRMED — cleanly monotonic, thousands of rows in every bucket.")


SIGNAL 2 — CTR vs position (behind FlyRank's CTR-fix logic / needs_ctr_fix)
                   n  mean_ctr
position_tier                 
top_3           2321     1.484
page_1         11814     0.652
striking        7304     0.323
page_3_5        7242     0.222
deep            1319     0.150
Verdict: CONFIRMED — cleanly monotonic, thousands of rows in every bucket.


Cleanly monotonic — every step down in position brings CTR down with it, and every bucket has thousands of rows behind it.

**Verdict: CONFIRMED.** This is the strongest, most trustworthy signal I have, which is why it's the backbone of the rule below rather than staleness alone.

**The rule, in plain words**

> A page is worth a CTR-fix review if it already has real search visibility, it ranks somewhere a reviewer could reasonably expect a decent click-through rate, its actual CTR is clearly below that (below the 0.5% line FlyRank's own `ctr_review_candidate` flag uses), *and* it hasn't been touched recently enough that a refresh is a plausible fix rather than noise.

- **Score** (readable on purpose): `score = stale * visible * position_ok * ctr_low * impressions_90d`
- **Reason code:** `stale_ctr_gap`
- **Action label:** `refresh_review`

Thresholds, and why: `impressions_90d >= 500` (visibility floor — matches FlyRank's own low-CTR flag), `0 < avg_position <= 20` (position where better CTR is realistic), `ctr < 0.5` (the CTR-fix line, confirmed above), `days_since_last_update >= 91` (the staleness cut that's actually backed by the MIXED check, not the noisy 180+ tail).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import os
import numpy as np

stale = df["days_since_last_update"] >= 91
visible = df["impressions_90d"] >= 500
position_ok = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
ctr_low = df["ctr"] < 0.5

flag = stale & visible & position_ok & ctr_low

df["score"] = np.where(flag, df["impressions_90d"], 0)
df["reason_code"] = np.where(flag, "stale_ctr_gap", "not_flagged")
df["action"] = np.where(flag, "refresh_review", "monitor")

queue_cols = [
    "content_id", "client_id", "score", "reason_code", "action",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update",
    "content_type", "main_intent",
]
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)[queue_cols]

os.makedirs("../outputs", exist_ok=True)
ranked.to_csv("../outputs/baseline_action_score.csv", index=False)

print(f"Rows written: {len(ranked)}")
print(f"Rows flagged (score > 0): {(ranked['score'] > 0).sum()}")
ranked.head(10)


Rows written: 30000
Rows flagged (score > 0): 3534


,content_id,client_id,score,reason_code,action,impressions_90d,avg_position,ctr,days_since_last_update,content_type,main_intent
0,content_5fe46e04994d,client_4e07408562,517715,stale_ctr_gap,refresh_review,517715,4.2,0.14,104,keyword article,informational
1,content_cb112fce36be,client_19581e27de,309910,stale_ctr_gap,refresh_review,309910,5.6,0.16,104,keyword article,transactional
2,content_36ff89c8214e,client_19581e27de,295097,stale_ctr_gap,refresh_review,295097,7.3,0.05,104,keyword article,informational
3,content_c21024970297,client_19581e27de,211366,stale_ctr_gap,refresh_review,211366,5.1,0.41,104,keyword article,commercial
4,content_c8e9d6ab9013,client_19581e27de,208678,stale_ctr_gap,refresh_review,208678,9.7,0.00,104,keyword article,informational
5,content_d17681677e69,client_19581e27de,201584,stale_ctr_gap,refresh_review,201584,5.8,0.24,104,keyword article,commercial
6,content_a7427266c305,client_19581e27de,201111,stale_ctr_gap,refresh_review,201111,5.7,0.11,104,keyword article,informational
7,content_c5063073d048,client_6208ef0f77,192205,stale_ctr_gap,refresh_review,192205,12.5,0.24,104,keyword article,informational
8,content_3d94572c3a35,client_19581e27de,190623,stale_ctr_gap,refresh_review,190623,4.3,0.24,104,keyword article,commercial
9,content_01908772c6db,client_19581e27de,187893,stale_ctr_gap,refresh_review,187893,4.0,0.45,104,keyword article,transactional


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked.head(10)

,content_id,client_id,score,reason_code,action,impressions_90d,avg_position,ctr,days_since_last_update,content_type,main_intent
0,content_5fe46e04994d,client_4e07408562,517715,stale_ctr_gap,refresh_review,517715,4.2,0.14,104,keyword article,informational
1,content_cb112fce36be,client_19581e27de,309910,stale_ctr_gap,refresh_review,309910,5.6,0.16,104,keyword article,transactional
2,content_36ff89c8214e,client_19581e27de,295097,stale_ctr_gap,refresh_review,295097,7.3,0.05,104,keyword article,informational
3,content_c21024970297,client_19581e27de,211366,stale_ctr_gap,refresh_review,211366,5.1,0.41,104,keyword article,commercial
4,content_c8e9d6ab9013,client_19581e27de,208678,stale_ctr_gap,refresh_review,208678,9.7,0.00,104,keyword article,informational
5,content_d17681677e69,client_19581e27de,201584,stale_ctr_gap,refresh_review,201584,5.8,0.24,104,keyword article,commercial
6,content_a7427266c305,client_19581e27de,201111,stale_ctr_gap,refresh_review,201111,5.7,0.11,104,keyword article,informational
7,content_c5063073d048,client_6208ef0f77,192205,stale_ctr_gap,refresh_review,192205,12.5,0.24,104,keyword article,informational
8,content_3d94572c3a35,client_19581e27de,190623,stale_ctr_gap,refresh_review,190623,4.3,0.24,104,keyword article,commercial
9,content_01908772c6db,client_19581e27de,187893,stale_ctr_gap,refresh_review,187893,4.0,0.45,104,keyword article,transactional


1. `content_5fe46e04994d` — **refresh_review**. 517,715 impressions at position 4.2 but 0.14% CTR — near the very top of page 1 and still barely clicked. *Wrong if:* the query is heavily zero-click (a featured snippet or AI overview is absorbing the clicks) rather than the page itself failing.
2. `content_cb112fce36be` — **refresh_review**. 309,910 impressions, position 5.6, CTR 0.16% — same story, high visibility with almost no return. *Wrong if:* this page recently changed URLs/tracking and the low CTR is a measurement gap, not a real one.
3. `content_36ff89c8214e` — **refresh_review**. Position 7.3, CTR 0.05% — the lowest CTR in the top 10 despite still being page-one. *Wrong if:* the title/meta mismatch the query intent so badly that a content refresh won't fix it — that's a metadata fix, not a refresh.
4. `content_c21024970297` — **refresh_review**. CTR 0.41%, closest to the 0.5% line of anyone here — a borderline case. *Wrong if:* 0.41% is actually normal for this query's intent (commercial queries often run lower CTR); the 0.5% cutoff may be too blunt across intents.
5. `content_c8e9d6ab9013` — **refresh_review**. CTR 0.00% at position 9.7 — literally zero clicks on ~200k impressions. *Wrong if:* this is a tracking/consent-mode gap (impressions logged, clicks not), not a content problem.
6. `content_d17681677e69` — **refresh_review**. Position 5.8, CTR 0.24%. *Wrong if:* a competitor's SERP feature (People Also Ask, a rich result) is sitting above this result and eating the clicks structurally.
7. `content_a7427266c305` — **refresh_review**. Position 5.7, CTR 0.11%. *Wrong if:* this page recently launched and the low CTR is still settling, not a stable pattern (worth checking last-30d vs prev-30d before acting).
8. `content_c5063073d048` — **refresh_review**. Position 12.5 (weakest position in the top 10), CTR 0.24% — this one may need a position push more than a CTR fix. *Wrong if:* the real fix here is ranking, and pushing on CTR (title/meta) won't move a page sitting outside the top 10.
9. `content_3d94572c3a35` — **refresh_review**. Position 4.3, CTR 0.24%. *Wrong if:* `days_since_last_update = 104` for this row is a batch/export artifact rather than a true last-edit date (see weak-picks check below) — then "stale" isn't really telling us anything here.
10. `content_01908772c6db` — **refresh_review**. CTR 0.45%, the highest CTR in the top 10 and just under the 0.5% line. *Wrong if:* this one's a false positive from the threshold being a hard cutoff rather than a curve — it's barely different from a page just above the line that got no flag at all.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Weak-pick pattern 1: client concentration in the top 10
print("Top-10 client concentration:")
print(ranked.head(10)["client_id"].value_counts())

# Weak-pick pattern 2: is days_since_last_update=104 a real spread, or one repeated value?
print("\nMost common exact days_since_last_update values, whole dataset:")
print(df["days_since_last_update"].value_counts().head(5))

# Leakage check: confirm none of the rule's inputs or the ranked queue's columns
# come from the label, the trend window, or a FlyRank product flag.
banned = ["trend_direction", "trend_pct", "is_declining_label",
          "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
          "health_score", "priority_score", "action_type", "refresh_tier"]
rule_inputs = ["days_since_last_update", "impressions_90d", "avg_position", "ctr"]
leaked = [c for c in rule_inputs if c in banned]
print(f"\nRule inputs: {rule_inputs}")
print(f"Any leaked/future-window/label-derived inputs used in the rule? {bool(leaked)}")

Top-10 client concentration:
client_id
client_19581e27de    8
client_4e07408562    1
client_6208ef0f77    1
Name: count, dtype: int64

Most common exact days_since_last_update values, whole dataset:
days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
Name: count, dtype: int64

Rule inputs: ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
Any leaked/future-window/label-derived inputs used in the rule? False


**What the checks turned up**

- **Client concentration.** 8 of the top 10 belong to a single client (`client_19581e27de`, 7,008 rows in the dataset — by far the largest client here). A reviewer using this queue as-is would spend their whole first pass on one account. *Fix if this were a real handoff:* cap picks per client (e.g. top 3 per client) before handing the queue to a human, or note the concentration explicitly so reviewers know to sample across clients too.
- **Suspicious exact-value clustering in staleness.** `days_since_last_update = 104` appears 8,773 times across the dataset (and covers most of the top 10). A value that common and that exact looks more like a batch export or bulk CMS update timestamp than 8,773 pages genuinely last touched on the same day. This doesn't invalidate the CTR-vs-position half of the rule, but it means the staleness half — even restricted to the MIXED-confirmed `>=91` range — may be measuring "which export batch this page belongs to" rather than true editorial staleness. Worth a note in the write-up, not a rule I'd trust blindly.
- **Leakage check.** The rule's four inputs (`days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`) are all trailing-90-day observed signals, all known at the decision moment. None of `trend_direction`, `trend_pct`, `is_declining_label`, the last-30d/prev-30d windows, or any FlyRank product flag (`health_score`, `priority_score`, etc. — which aren't even in this dataset) went into the score, reason code, or action label. `is_declining_label` was used once, above, only to check the staleness signal — never as a rule input.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.